In [1]:
#importing the necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import pickle
from math import pi

In [228]:
#loading the datasets
train_df=pd.read_csv("train.csv")
store_df = pd.read_csv("store.csv")
test_df = pd.read_csv("test.csv")


/var/folders/n1/w1_yvs_x57l_7v6xy03xjqkm0000gn/T/ipykernel_88781/1119007227.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df=pd.read_csv("train.csv")


#Data Preview

In [229]:
#preview the train dataset
train_df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [230]:
#preview store dataset
store_df.head(10)

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN
5,6,a,a,310.0,12.0,2013.0,0,NaN,NaN,NaN
6,7,a,c,24000.0,4.0,2013.0,0,NaN,NaN,NaN
7,8,a,a,7520.0,10.0,2014.0,0,NaN,NaN,NaN
8,9,a,c,2030.0,8.0,2000.0,0,NaN,NaN,NaN
9,10,a,a,3160.0,9.0,2009.0,0,NaN,NaN,NaN


In [231]:
#data information
train_df.info()
print("------------------------------")
store_df.info()
print("------------------------------")
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  object
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(2)
memory usage: 69.8+ MB
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115

In [232]:
#checking the shape of our train dataset
print(train_df.shape)
print(store_df.shape)
print(test_df.shape)

(1017209, 9)
(1115, 10)
(41088, 8)


Data cleaning

In [233]:
#checking to see if we have missing data
train_df.isnull().sum()
#we don't have any missing data

Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

In [234]:
#checking to see if our data is duplicated
train_df.duplicated().any()
#we don't have any duplicates

False

In [235]:
#convert date to datetime
train_df.Date = pd.to_datetime(train_df['Date'])

In [236]:
#checking unique values
def unique_cols(train_df):
  unique_dict = {}
  cat_cols = [
        'DayOfWeek', 'Open', 'Promo',
        'StateHoliday', 'SchoolHoliday']
  for col in cat_cols:
    unique_dict[col] = train_df[col].unique()
  return unique_dict

print(unique_cols(train_df))

#from the unique values, we notice that we have 0 as a string '0' and as an integer 0.

{'DayOfWeek': array([5, 4, 3, 2, 1, 7, 6]), 'Open': array([1, 0]), 'Promo': array([1, 0]), 'StateHoliday': array(['0', 'a', 'b', 'c', 0], dtype=object), 'SchoolHoliday': array([1, 0])}


In [ ]:
#clean StateHoliday to remove the integer 0 and make it a string
train_df.StateHoliday = train_df['StateHoliday'].astype(str)
#checking to see if it worked
train_df.StateHoliday.unique()

In [238]:
# Dropping rows where the store was closed, and dropping the Open column since it is now irrelevant
train_df = train_df[train_df['Open'] == 1].drop(columns=['Open']).copy()

In [239]:
#Filtering out records where Sales were 0
train_df= train_df[train_df['Sales'] > 0].copy()

cleaning stores data

In [240]:
#checking unique values
def unique_cols(train):
  unique_dict = {}
  cat_cols = [
        'PromoInterval', 'StoreType', 'Assortment',
        'Promo2']
  for col in cat_cols:
    unique_dict[col] = store_df[col].unique()
  return unique_dict

print(unique_cols(store_df))

{'PromoInterval': array([nan, 'Jan,Apr,Jul,Oct', 'Feb,May,Aug,Nov', 'Mar,Jun,Sept,Dec'],
      dtype=object), 'StoreType': array(['c', 'a', 'd', 'b'], dtype=object), 'Assortment': array(['a', 'c', 'b'], dtype=object), 'Promo2': array([0, 1])}


In [241]:
#percentage of missing values
store_df.isnull().mean() * 100

Store                         0.000000
StoreType                     0.000000
Assortment                    0.000000
CompetitionDistance           0.269058
CompetitionOpenSinceMonth    31.748879
CompetitionOpenSinceYear     31.748879
Promo2                        0.000000
Promo2SinceWeek              48.789238
Promo2SinceYear              48.789238
PromoInterval                48.789238
dtype: float64

In [242]:
#imputing competition distance with median
store_df['CompetitionDistance_Missing'] = store_df['CompetitionDistance'].isnull().astype(int)
store_df['CompetitionDistance'] = store_df['CompetitionDistance'].fillna(store_df['CompetitionDistance'].median())


In [243]:
# Checking to see if null values in rows in CompetitionOpenSinceMonth and CompetitionOpenSinceYear overlap
null_rows = store_df[store_df['CompetitionOpenSinceYear'].isnull() | store_df['CompetitionOpenSinceMonth'].isnull()]

same_rows = (store_df['CompetitionOpenSinceYear'].isnull() == store_df['CompetitionOpenSinceMonth'].isnull()).all()
print(same_rows)

True


since CompetitionOpenSinceYear and CompetitionOpenSinceMonth has an overlap in the null values, then we can fill them by default year and month.Since the dataset is from 2013 to 2015,we will use 2013.and 1 to represent january the earliest point of the year.




In [244]:
#imputing CompetitionOpenSinceYear and CompetitionOpenSinceMonth
store_df['CompetitionOpenSinceYear'] = store_df['CompetitionOpenSinceYear'].fillna(2013).astype(int)
store_df['CompetitionOpenSinceMonth'] = store_df['CompetitionOpenSinceMonth'].fillna(1).astype(int)


In [245]:
# checking to see if Promo2SinceYear,Promo2SinceWeek and PromoInterval are null when Promo2 is 0,
# meaning store is not participating

conflict_rows = store_df[(store_df['Promo2'] == 0) & (
    store_df['Promo2SinceYear'].notnull() |
    store_df['Promo2SinceWeek'].notnull() |
    store_df['PromoInterval'].notnull()
)]
print(conflict_rows.shape[0])

0


In [246]:
#we will impute 0 of date related columns with 0
d_rel= ['Promo2SinceWeek', 'Promo2SinceYear']
for col in d_rel:
  store_df[col] = store_df[col].fillna(0)
  store_df[col] = store_df[col].astype(int)

In [247]:
#Imputing PromoInterval
store_df['PromoInterval'] = store_df['PromoInterval'].fillna('None')


In [248]:
#rechecking to see if we have any missing values
store_df.isnull().sum()

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            0
CompetitionOpenSinceMonth      0
CompetitionOpenSinceYear       0
Promo2                         0
Promo2SinceWeek                0
Promo2SinceYear                0
PromoInterval                  0
CompetitionDistance_Missing    0
dtype: int64

In [249]:
#checking to see if our data is duplicated
store_df.duplicated().any()
#we don't have any duplicates

False

cleaning test data

In [250]:
test_df['Date'] = pd.to_datetime(test_df['Date'])

In [ ]:
#clean StateHoliday to remove the integer 0 and make it a string
test_df.StateHoliday = test_df['StateHoliday'].astype(str)
#checking to see if it worked
test_df.StateHoliday.unique()

In [ ]:
test_df['Open'].fillna(1, inplace=True)
test_df = test_df[test_df['Open'] == 1].drop(columns=['Open']).copy()

feature engineering

In [ ]:
# extracting date features
def date_features(df):
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
    df['DayOfYear'] = df['Date'].dt.dayofyear


    # df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
    # df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)
    # df['IsWeekend'] = (df['Date'].dt.weekday >= 5).astype(int)

    # # Cyclical Encoding, this part was done after thr first preprocessing to see if the model's performanc ewill improve
    # df['DayOfWeek_sin'] = np.sin(2 * pi * df['DayOfWeek'] / 7)
    # df['DayOfWeek_cos'] = np.cos(2 * pi * df['DayOfWeek'] / 7)
    # df['Month_sin'] = np.sin(2 * pi * df['Month'] / 12)
    # df['Month_cos'] = np.cos(2 * pi * df['Month'] / 12)
    # df['DayOfYear_sin'] = np.sin(2 * pi * df['DayOfYear'] / 366)
    # df['DayOfYear_cos'] = np.cos(2 * pi * df['DayOfYear'] / 366)

    # # Dropping original cyclical columns
    # df = df.drop(columns=['DayOfWeek', 'WeekOfYear', 'DayOfYear'])
    # return df

train_df = date_features(train_df)
test_df = date_features(test_df)


In [254]:
#dropping Date column since it is irrelevant
train_df = train_df.drop(columns=['Date'])
test_df = test_df.drop(columns=['Date'])

In [ ]:
# Merging the datasets on 'Store'

sales_m= pd.merge(train_df, store_df, on='Store', how='left')
test_m = pd.merge(test_df, store_df, on='Store', how='left')


In [ ]:
#checking for missing values in our test merged dataset
print(test_m.isnull().any())
print(sales_m.isnull().any())

In [258]:

test_m['Sales'] = 0.0
test_m['Customers'] = 0.0

In [259]:
#sorting data by store,year,month and column
sales_m = sales_m.sort_values(['Store', 'Year','Month', 'Day']).copy()
test_m = test_m.sort_values(['Store', 'Year', 'Month','Day']).copy()

This was further preprocessing to see if including lags and windows to give the model more information about past trends since this is a time series kind of data, the model would improve.

In [ ]:
# #creating lags and windows 
# LAGS = [7, 14, 28]  
# for lag in LAGS:
#     sales_m[f'Sales_Lag_{lag}'] = sales_m.groupby('Store')['Sales'].shift(lag)
#     sales_m[f'Customers_Lag_{lag}'] = sales_m.groupby('Store')['Customers'].shift(lag)
#     test_m[f'Sales_Lag_{lag}'] = test_m.groupby('Store')['Sales'].shift(lag)

# WINDOWS = [7, 28]
# for window in WINDOWS:
#     sales_m[f'Sales_RollingMean_{window}'] = sales_m.groupby('Store')['Sales'].shift(1).rolling(window=window, min_periods=1).mean()
#     sales_m[f'Customers_RollingMean_{window}'] = sales_m.groupby('Store')['Customers'].shift(1).rolling(window=window, min_periods=1).mean()
#     test_m[f'Sales_RollingMean_{window}'] = test_m.groupby('Store')['Sales'].shift(1).rolling(window=window, min_periods=1).mean()


In [261]:
num_cols = [
    'CompetitionDistance', 'CompetitionOpenMonths', 'Promo2SinceWeek', 'Promo2SinceYear',
    'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Lag_28', 'Sales_RollingMean_7', 'Sales_RollingMean_28',
    'Year', 'Day','DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'DayOfYear_sin', 'DayOfYear_cos','Month'
]

In [ ]:
# #identifying lag columns and filling them with rolling mean.
# lag_cols = [col for col in sales_m.columns if 'Lag' in col or 'RollingMean' in col]

# for col in lag_cols:
#     sales_m[col] = sales_m[col].fillna(sales_m[col].mean())

# for col in [c for c in num_cols if 'Sales_Lag' in c or 'Sales_RollingMean' in c]:
#     test_m[col] = test_m[col].fillna(sales_m[col].mean())

In [ ]:
# #calculating months the competition was on.
# sales_m['CompetitionOpenMonths'] = (sales_m['Year'] - sales_m['CompetitionOpenSinceYear']) * 12 + \
#                                    (sales_m['Month'] - sales_m['CompetitionOpenSinceMonth'])
# test_m['CompetitionOpenMonths'] = (test_m['Year'] - test_m['CompetitionOpenSinceYear']) * 12 + \
#                                    (test_m['Month'] - test_m['CompetitionOpenSinceMonth'])

# sales_m['CompetitionOpenMonths'] = sales_m['CompetitionOpenMonths'].clip(lower=0)
# test_m['CompetitionOpenMonths'] = test_m['CompetitionOpenMonths'].clip(lower=0)

In [ ]:
# identifying is a promo is active for a specifc month in a given store
#this was also done a spart of preprocessing to see if it will improve model performance.
# def promo2_active(row):
#     # if row['Promo2'] == 0 or row['Promo2SinceYear'] == 0:
#     #     return 0
    
#     promo_start = pd.Timestamp(year=row['Promo2SinceYear'], month=1, day=1) + pd.to_timedelta((row['Promo2SinceWeek']-1)*7, unit='d')
#     current_date = pd.Timestamp(year=row['Year'], month=row['Month'], day=row['Day'])
    
#     if current_date >= promo_start:
#         if row['PromoInterval'] != 'None':
#             promo_months = row['PromoInterval'].split(',')
#             month_cur = pd.Timestamp(year=row['Year'], month=row['Month'], day=1).strftime('%b')
#             return int(month_cur in promo_months)
#         else:
            
#             return 1
#     return 0

# sales_m['Promo2Active'] = sales_m.apply(promo2_active, axis=1)
# test_m['Promo2Active'] = test_m.apply(promo2_active, axis=1)

# sales_m['Promo2Season'] = sales_m['Promo2Active'] * sales_m['Month']
# test_m['Promo2Season'] = test_m['Promo2Active'] * test_m['Month']

In [265]:
#log transforming sales for modeling since it is a time series tyoe of data
sales_m['Sales'] = np.log1p(sales_m['Sales'])
sales_m = sales_m.drop(columns=['Customers'])

In [ ]:
#getting our categorical columns
cat_cols= ['Store', 'StoreType', 'Assortment', 'PromoInterval', 'StateHoliday']


In [267]:
#encoding categorical columns
encoders = {}
for col in cat_cols:
    sales_m[col], uniques = pd.factorize(sales_m[col])
    encoder = LabelEncoder()
    encoder.fit(uniques)
    encoders[col] = encoder
    test_m[col] = test_m[col].apply(lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1)

In [ ]:
#scaling data and saving it
scaler = StandardScaler()
sales_m[num_cols] = scaler.fit_transform(sales_m[num_cols])
test_m[num_cols] = scaler.transform(test_m[num_cols])

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

scaled_cols_for_fitting = num_cols
with open("scaler_cols.pkl", "wb") as f:
    pickle.dump(scaled_cols_for_fitting, f)




In [270]:
test_m = test_m.drop(columns=['Sales', 'Customers'])

In [ ]:
#creating the final columns for modeling
final_cols = [col for col in sales_m.columns if col not in ['Sales']]
sales_e = sales_m[final_cols + ['Sales']]
test_e = test_m[['Id'] + final_cols]

In [ ]:
# # Save train columns for later use
# with open("train_cols.pkl", "wb") as f:
#     pickle.dump(train_cols, f)

In [274]:
#saving cleaned data to csv files
sales_e.to_csv('train_cl.csv', index=False)
test_e.to_csv('test_cl.csv', index=False)